In [ ]:
import numpy as np


def argmin_points(f, X, axis=0, return_value=False):
    """
    Encuentra el punto de X que minimiza f.

    Parámetros
    ----------
    f : callable
        Función objetivo. Debe recibir un punto x ∈ R^d.

    X : array-like
        Conjunto de puntos.

    axis : int, default=0
        - axis=0: cada columna de X es un punto.
        - axis=1: cada fila de X es un punto.

    return_value : bool, default=False
        Si True, retorna también el valor mínimo de f.

    Retorna
    -------
    x_min : ndarray
        Punto que minimiza f sobre X.

    Si return_value=True:
        (x_min, f_min)
    """

    X = np.asarray(X, dtype=float)

    if X.ndim == 1:
        # Un solo punto
        X_points = X.reshape(1, -1)

    elif X.ndim == 2:
        if axis == 0:
            # Columnas = puntos
            X_points = X.T
        elif axis == 1:
            # Filas = puntos
            X_points = X
        else:
            raise ValueError("axis debe ser 0 o 1.")

    else:
        raise ValueError(
            "X debe ser un arreglo unidimensional o bidimensional."
        )

    # Evaluamos f en cada punto
    values = np.array([f(x) for x in X_points])

    if values.ndim != 1:
        raise ValueError(
            "La función f debe retornar un único valor escalar "
            "para cada punto."
        )

    # Verificar valores inválidos
    if not np.all(np.isfinite(values)):
        raise ValueError(
            "f produjo valores NaN o infinitos."
        )

    # Índice del mínimo
    i = np.argmin(values)

    x_min = X_points[i]

    if return_value:
        return x_min, values[i]

    return x_min

def f_oracle(xk, z , grad_phi, exp_inv):
    return grad_phi(xk)@exp_inv(xk,z)

def oracle(xk, z , grad_phi, exp_inv,X):
    return argmin_points(f_oracle(xk, z , grad_phi, exp_inv), X, axis=1)
def gamma(t, a,b):
    # retorna la geodesia entre dos puntos de la esfera unitaria
    # donde inicia en a y terminar en b
def rfw_g_convex(x0, X, gamma, grad_phi, exp_inv, max_iter = 1000, tol = 1e-6):
    xk  = x0
    for k in range(max_iter):
        zk = oracle(xk, z , grad_phi, exp_inv,X)
        sk = 1/(k/2+1)

In [14]:
# Gemini

import numpy as np
from scipy.optimize import nnls

def proj_Q_sphere(u, H):
    """
    Proyecta u sobre Q \cap S_n donde Q = cone(H)[cite: 2, 14, 15].
    Utiliza Mínimos Cuadrados No Negativos (NNLS).
    """
    y_star, _ = nnls(H, u)
    w = H @ y_star
    norm_w = np.linalg.norm(w)
    
    if norm_w > 1e-12:
        v_bar = w / norm_w
    else:
        # Si la proyección es 0, buscar el generador de H con máximo producto interno[cite: 15]
        j_max = np.argmax(H.T @ u)
        v_bar = H[:, j_max] / np.linalg.norm(H[:, j_max])
        
    f_val = np.dot(u, v_bar)
    return v_bar, f_val

def exp_map_inv(u, z):
    """
    Mapa exponencial inverso Exp_u^{-1}(z) en S^{n-1}.
    """
    dot_prod = np.clip(np.dot(u, z), -1.0, 1.0)
    theta = np.arccos(dot_prod)
    if theta < 1e-12:
        return np.zeros_like(u), 0.0
    
    sin_theta = np.sin(theta)
    tangent_vec = (theta / sin_theta) * (z - dot_prod * u)
    return tangent_vec, theta

def geodesic_step(u, z, theta, s):
    """
    Avanza una fracción s a lo largo de la geodésica de u a z en S^{n-1}.
    """
    if theta < 1e-12 or s < 1e-12:
        return u
    
    sin_theta = np.sin(theta)
    u_next = (np.sin((1.0 - s) * theta) / sin_theta) * u + (np.sin(s * theta) / sin_theta) * z
    return u_next / np.linalg.norm(u_next)

def rfw_maxmin_angle(G, H, tol=1e-4, max_iter=200):
    """
    Algoritmo de Frank-Wolfe Riemaniano para resolver min_{u in P \cap S_n} F_Q(u)[cite: 2, 8, 26, 31].
    
    Parámetros:
        G: np.ndarray (n, p) - Matriz cuyos coeficientes/columnas son generadores de P[cite: 14].
        H: np.ndarray (n, q) - Matriz cuyos coeficientes/columnas son generadores de Q[cite: 14].
        tol: Criterio de parada sobre la brecha de dualidad (FW gap).
        max_iter: Número máximo de iteraciones.
        
    Retorna:
        s_opt: Coseno del ángulo máx-min (s(P, Q))[cite: 8].
        angle_rad: Ángulo máx-min en radianes (Theta(P, Q))[cite: 8].
        u_opt: Vector óptimo en P \cap S_n.
    """
    # Normalización de los generadores de G y H[cite: 14]
    G = G / np.linalg.norm(G, axis=0, keepdims=True)
    H = H / np.linalg.norm(H, axis=0, keepdims=True)
    
    p = G.shape[1]
    
    # 1. Inicialización en uno de los vértices extremales de P
    u = G[:, 0].copy()
    
    best_u = u.copy()
    best_f = 1.0
    
    for k in range(max_iter):
        # 2. Proyección y Subgradiente Euclidiano
        v_bar, f_val = proj_Q_sphere(u, H)
        
        if f_val < best_f:
            best_f = f_val
            best_u = u.copy()
            
        # 3. Subgradiente Riemanniano g_k en T_u S^{n-1}
        g_k = v_bar - f_val * u
        
        # 4. Oráculo Lineal Riemanniano (RLO) sobre P \cap S_n[cite: 31]
        best_rlo_val = np.inf
        z_k = None
        best_theta = 0.0
        
        for j in range(p):
            g_j = G[:, j]
            t_vec, theta = exp_map_inv(u, g_j)
            rlo_val = np.dot(g_k, t_vec)
            
            if rlo_val < best_rlo_val:
                best_rlo_val = rlo_val
                z_k = g_j
                best_theta = theta
                
        # Brecha de dualidad Riemanniana (FW gap)
        fw_gap = -best_rlo_val
        if fw_gap <= tol:
            break
            
        # 5. Tamaño de paso Riemanniano[cite: 31]
        s_k = 2.0 / (k + 2.0)
        
        # 6. Actualización a lo largo de la geodésica[cite: 31]
        u = geodesic_step(u, z_k, best_theta, s_k)

    s_opt = best_f
    angle_rad = np.arccos(np.clip(s_opt, -1.0, 1.0))
    
    return s_opt, angle_rad, best_u



In [15]:
def matriz_schur_identidad(n):

  ident = np.eye(n)

  schur = ident[:,0] - ident[:,1]

  for i in range(1,n-1):

    schur = np.vstack((schur, ident[:,i] - ident[:,i+1]))

  schur = schur.T

  return schur/np.linalg.norm(schur, axis = 0 , keepdims=True), np.eye(n)

In [16]:

# ==========================================
# Ejemplo de uso con conos aleatorios en R^3
# ==========================================
if __name__ == "__main__":
    s , id = matriz_schur_identidad(3)
    
    # Generar matrices aleatorias de generadores[cite: 14, 18]
    G_mat = id
    H_mat = s
    
    # Resolver problema s(P, Q)
    s_PQ, theta_PQ, u_star = rfw_maxmin_angle(G_mat, H_mat)
    
    print(f"Resultado s(P, Q) = cos(Theta(P, Q)): {s_PQ:.6f}")
    print(f"Ángulo Máx-Min Theta(P, Q) (rad): {theta_PQ:.6f}")
    print(f"Ángulo Máx-Min Theta(P, Q) (grados): {np.degrees(theta_PQ):.2f}°")
    print(f"Solución u*: {u_star}")

Resultado s(P, Q) = cos(Theta(P, Q)): -0.316217
Ángulo Máx-Min Theta(P, Q) (rad): 1.892536
Ángulo Máx-Min Theta(P, Q) (grados): 108.43°
Solución u*: [0.         0.44722367 0.89442216]


In [1]:
# rfw_maxmin_angle.py

import numpy as np
from scipy.optimize import nnls, minimize

_EPS = 1e-10


# ======================================================
# Utilidades básicas
# ======================================================

def _as_matrix(A):
    A = np.asarray(A, dtype=float)
    if A.ndim == 1:
        A = A.reshape(-1, 1)
    return A


def normalize_columns(A):
    """
    Normaliza las columnas de A y elimina columnas nulas.
    """
    A = _as_matrix(A)
    norms = np.linalg.norm(A, axis=0)
    keep = norms > _EPS
    if not np.any(keep):
        raise ValueError("Todos los generadores son nulos.")
    A = A[:, keep]
    norms = norms[keep]
    return A / norms[None, :]


# ======================================================
# Geometría riemanniana en la esfera
# ======================================================

def exp_sphere(u, eta, eps=_EPS):
    """
    Exponential map en la esfera.

    u   : punto en la esfera
    eta : vector tangente en T_u S
    """
    u = u / np.linalg.norm(u)
    eta = eta - np.dot(eta, u) * u
    theta = np.linalg.norm(eta)

    if theta < eps:
        return u.copy()

    if theta > np.pi:
        eta = eta * (np.pi / theta)
        theta = np.pi

    y = np.cos(theta) * u + np.sin(theta) * eta / theta
    return y / np.linalg.norm(y)


def log_sphere(u, z, eps=_EPS):
    """
    Inverse exponential map en la esfera.

    Devuelve eta = Log_u(z) en T_u S.
    """
    u = u / np.linalg.norm(u)
    z = z / np.linalg.norm(z)

    c = float(np.dot(u, z))
    c = np.clip(c, -1.0, 1.0)

    if c >= 1.0 - eps:
        return np.zeros_like(u)

    if c <= -1.0 + eps:
        # Caso antipodal: Log no es único.
        # Elegimos una dirección tangente determinística.
        w = np.zeros_like(u)
        idx = int(np.argmin(np.abs(u)))
        w[idx] = 1.0
        w = w - np.dot(w, u) * u
        nw = np.linalg.norm(w)

        if nw < eps:
            rng = np.random.default_rng(0)
            w = rng.standard_normal(u.shape)
            w = w - np.dot(w, u) * u
            nw = np.linalg.norm(w)

        return np.pi * w / nw

    denom = np.sqrt(max(eps, 1.0 - c * c))
    factor = np.arccos(c) / denom
    eta = factor * (z - c * u)
    eta = eta - np.dot(eta, u) * u

    return eta


# ======================================================
# Proyección sobre un cono poliédrico y sobre cono ∩ esfera
# ======================================================

def proj_cone_x(u, G, eps=_EPS):
    """
    Proyección euclídea aproximada de u sobre el cono cone(G).

    Devuelve:
      p : proyección sobre el cono
      x : coeficientes no negativos tales que p ≈ G @ x

    Se usa NNLS:
        min_x ||G x - u||_2^2  s.t. x >= 0.
    """
    vals = G.T @ u

    if vals.size == 0 or np.max(vals) <= eps:
        return np.zeros_like(u), np.zeros(G.shape[1])

    x, _ = nnls(G, u)
    p = G @ x

    if np.linalg.norm(p) <= eps:
        j = int(np.argmax(vals))
        x = np.zeros(G.shape[1])
        x[j] = max(0.0, vals[j])
        p = G @ x

    return p, x


def proj_cone(u, G, eps=_EPS):
    """
    Proyección euclídea sobre el cono poliédrico cone(G).
    """
    p, _ = proj_cone_x(u, G, eps=eps)
    return p


def proj_cone_sphere(u, G, eps=_EPS):
    """
    Devuelve un punto en Proj_{cone(G) ∩ S}(u).

    Para conos poliédricos, usamos la caracterización del artículo:
      si max(H^T u) > 0  -> Proj_Q(u)/||Proj_Q(u)||
      si max(H^T u) = 0  -> cone(active) ∩ S
      si max(H^T u) < 0  -> generador activo
    """
    u = u / np.linalg.norm(u)
    vals = G.T @ u
    maxv = np.max(vals)

    if maxv > eps:
        p = proj_cone(u, G, eps=eps)
        np_p = np.linalg.norm(p)
        if np_p > eps:
            return p / np_p

    if abs(maxv) <= eps:
        J = np.where(vals >= maxv - eps)[0]
        if J.size > 0:
            w = np.sum(G[:, J], axis=1)
            nw = np.linalg.norm(w)
            if nw > eps:
                return w / nw
            return G[:, J[0]].copy()

    j = int(np.argmax(vals))
    return G[:, j].copy()


def support_value(u, G, eps=_EPS):
    """
    Evalúa f(u) = max_{v in cone(G) ∩ S} <u, v>.

    Devuelve:
      f : valor de la función soporte
      v : un maximizador/proyección asociada
    """
    u = u / np.linalg.norm(u)
    v = proj_cone_sphere(u, G, eps=eps)
    return float(np.dot(u, v)), v


def riemannian_subgradient(u, H, eps=_EPS):
    """
    Subgradiente riemanniano de f(u)=F_H(u) en la esfera.

    Si v ∈ Proj_{H∩S}(u), entonces un subgradiente euclídeo
    de la función soporte es v. El subgradiente riemanniano es
    la proyección tangente:

        xi = v - <u, v> u.
    """
    u = u / np.linalg.norm(u)
    v = proj_cone_sphere(u, H, eps=eps)
    xi = v - np.dot(u, v) * u
    xi = xi - np.dot(xi, u) * u
    return xi, v


# ======================================================
# Oráculo lineal riemanniano
# ======================================================

def _oracle_objective(u, xi, z):
    """
    Valor del oráculo:
        <xi, Log_u(z)>
    """
    return float(np.dot(xi, log_sphere(u, z)))


def linear_oracle(u, xi, G, mode="projection", local_maxiter=100, eps=_EPS):
    """
    Oráculo lineal riemanniano aproximado.

    Resuelve aproximadamente:

        min_{z in cone(G) ∩ S} <xi, Log_u(z)>.

    Modos:
      'generators' : busca sólo entre generadores de G y u.
      'projection' : agrega la proyección de -xi sobre cone(G).
      'local'      : refina con un solver NLP sobre pesos x >= 0.
    """
    u = u / np.linalg.norm(u)
    xi = xi - np.dot(xi, u) * u

    candidates = [u.copy()]

    # Generadores de P
    for j in range(G.shape[1]):
        candidates.append(G[:, j].copy())

    # Candidato basado en proyección de -xi sobre P
    if mode in {"projection", "local"} and np.linalg.norm(xi) > eps:
        p, _ = proj_cone_x(-xi, G, eps=eps)
        np_p = np.linalg.norm(p)
        if np_p > eps:
            candidates.append(p / np_p)

    best_z = None
    best_val = np.inf

    for z in candidates:
        nz = np.linalg.norm(z)
        if nz <= eps:
            continue
        z = z / nz
        val = _oracle_objective(u, xi, z)
        if val < best_val:
            best_val = val
            best_z = z.copy()

    if best_z is None:
        best_z = u.copy()
        best_val = 0.0

    if mode in {"generators", "projection"}:
        return best_z, best_val

    if mode == "local":
        return _local_oracle(
            u=u,
            xi=xi,
            G=G,
            z_init=best_z,
            val_init=best_val,
            maxiter=local_maxiter,
            eps=eps,
        )

    raise ValueError("mode debe ser 'generators', 'projection' o 'local'.")


def _local_oracle(u, xi, G, z_init, val_init, maxiter=100, eps=_EPS):
    """
    Refinamiento local del oráculo.

    Parametrizamos z = Gx / ||Gx|| con x >= 0.
    """
    q = G.shape[1]
    x_starts = []

    # 1. Mejores generadores según el objetivo del oráculo
    gen_vals = np.array([
        _oracle_objective(u, xi, G[:, j])
        for j in range(q)
    ])
    topn = min(q, 5)
    top_idx = np.argsort(gen_vals)[:topn]

    for j in top_idx:
        x = np.zeros(q)
        x[j] = 1.0
        x_starts.append(x)

    # 2. Proyección de -xi
    if np.linalg.norm(xi) > eps:
        p, x_proj = proj_cone_x(-xi, G, eps=eps)
        if np.linalg.norm(p) > eps:
            x_starts.append(x_proj / np.linalg.norm(p))

    # 3. Representación aproximada de z_init
    xz, _ = nnls(G, z_init)
    pz = G @ xz
    nz = np.linalg.norm(pz)
    if nz > eps:
        x_starts.append(xz / nz)

    # 4. Punto uniforme
    x_unif = np.ones(q)
    gx_unif = G @ x_unif
    ngx = np.linalg.norm(gx_unif)
    if ngx > eps:
        x_starts.append(x_unif / ngx)

    def obj_x(x):
        gx = G @ x
        ng = np.linalg.norm(gx)
        if ng <= eps:
            return 1e20
        z = gx / ng
        return float(np.dot(xi, log_sphere(u, z)))

    cons = (
        {
            "type": "eq",
            "fun": lambda x: float(np.dot(G @ x, G @ x) - 1.0),
            "jac": lambda x: 2.0 * (G.T @ (G @ x)),
        },
    )

    bounds = [(0.0, None)] * q

    best_z = z_init.copy()
    best_val = val_init

    for x0 in x_starts:
        gx0 = G @ x0
        ng0 = np.linalg.norm(gx0)
        if ng0 <= eps:
            continue

        x0 = x0 / ng0

        try:
            res = minimize(
                obj_x,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=cons,
                options={
                    "maxiter": maxiter,
                    "ftol": 1e-6,
                    "disp": False,
                },
            )

            x = np.clip(res.x, 0.0, None)
            gx = G @ x
            ng = np.linalg.norm(gx)

            if ng > eps:
                x = x / ng
                z = G @ x
                val = obj_x(x)

                if val < best_val:
                    best_val = val
                    best_z = z.copy()

        except Exception:
            pass

    return best_z, best_val


# ======================================================
# Inicialización tipo warm-start
# ======================================================

def best_initial_generator(G, H):
    """
    Warm-start: elegir el generador de P que minimiza F_Q.
    """
    best_u = None
    best_f = np.inf

    for j in range(G.shape[1]):
        u = G[:, j]
        f, _ = support_value(u, H)
        if f < best_f:
            best_f = f
            best_u = u.copy()

    return best_u, best_f


# ======================================================
# Riemannian Frank-Wolfe para el ángulo max-min
# ======================================================

def solve_maxmin_angle_rfw(
    G,
    H,
    mode="projection",
    max_iter=200,
    tol=1e-5,
    verbose=False,
    project_to_P=False,
    local_maxiter=100,
):
    """
    Resuelve aproximadamente:

        min_{u in cone(G) ∩ S} F_H(u),

    donde F_H(u) = max_{v in cone(H) ∩ S} <u, v>.

    Devuelve:
      theta : ángulo max-min Θ(P, Q)
      cos   : cos(theta)
      u     : mejor punto encontrado
      history: historial de iteraciones
    """
    G = normalize_columns(G)
    H = normalize_columns(H)

    # Warm-start
    u0, f0 = best_initial_generator(G, H)
    u = u0.copy()

    best_u = u.copy()
    best_f = f0

    history = []

    for k in range(max_iter):
        f_current, _ = support_value(u, H)

        if f_current < best_f:
            best_f = f_current
            best_u = u.copy()

        xi, _ = riemannian_subgradient(u, H)
        nxi = np.linalg.norm(xi)

        # Si el subgradiente tangente es casi cero, estamos en un punto
        # first-order flat. Podemos parar o intentar usar el mejor generador.
        if nxi < _EPS:
            if f0 < f_current - 10.0 * tol:
                u = u0.copy()
                continue
            break

        z, lin_val = linear_oracle(
            u=u,
            xi=xi,
            G=G,
            mode=mode,
            local_maxiter=local_maxiter,
        )

        # FW gap aproximado
        gap = max(0.0, -lin_val)

        history.append(
            {
                "iter": k,
                "f": f_current,
                "gap": gap,
                "norm_xi": nxi,
            }
        )

        if verbose:
            print(
                f"iter={k:4d} | f={f_current:.8f} | "
                f"gap={gap:.3e} | ||xi||={nxi:.3e}"
            )

        if gap <= tol:
            break

        alpha = 2.0 / (k + 2.0)

        eta = log_sphere(u, z)
        u_new = exp_sphere(u, alpha * eta)

        if project_to_P:
            p, _ = proj_cone_x(u_new, G)
            np_p = np.linalg.norm(p)
            if np_p > _EPS:
                u = p / np_p
            else:
                u = u_new / np.linalg.norm(u_new)
        else:
            u = u_new / np.linalg.norm(u_new)

    # Evaluación final
    f_best_final, _ = support_value(best_u, H)
    best_f = min(best_f, f_best_final)

    theta = float(np.arccos(np.clip(best_f, -1.0, 1.0)))

    return {
        "theta": theta,
        "cos": float(best_f),
        "u": best_u,
        "history": history,
    }


# ======================================================
# Distancia Dis_r(P, Q)
# ======================================================

def dissimilarity_dis_r(G, H, r=2, **kwargs):
    """
    Calcula la distancia

        Dis_r(P, Q)
        = 2 ||( sin(Θ(P,Q)/2), sin(Θ(Q,P)/2) )||_r.

    G : generadores de P
    H : generadores de Q
    r : 1, 2, o np.inf
    """
    res_PQ = solve_maxmin_angle_rfw(G, H, **kwargs)
    res_QP = solve_maxmin_angle_rfw(H, G, **kwargs)

    s1 = np.sin(res_PQ["theta"] / 2.0)
    s2 = np.sin(res_QP["theta"] / 2.0)

    if r == np.inf:
        dist = 2.0 * max(s1, s2)
    else:
        dist = 2.0 * float(np.linalg.norm(np.array([s1, s2]), ord=r))

    return dist, res_PQ, res_QP

In [2]:
def matriz_schur_identidad(n):

  ident = np.eye(n)

  schur = ident[:,0] - ident[:,1]

  for i in range(1,n-1):

    schur = np.vstack((schur, ident[:,i] - ident[:,i+1]))

  schur = schur.T

  return schur/np.linalg.norm(schur, axis = 0 , keepdims=True), np.eye(n)

In [13]:
s , id = matriz_schur_identidad(6)
    
    # Generar matrices aleatorias de generadores[cite: 14, 18]
G = id
H = s

dist, res_PQ, res_QP = dissimilarity_dis_r(
        G,
        H,
        r=2,
        mode="projection",   # o "generators" / "local"
        max_iter=100,
        tol=1e-4,
        verbose=False,
    )

print("\nResultado")
print("Dis_2(P,Q) =", dist)
print("Theta(P,Q) =", res_PQ["theta"])
print("Theta(Q,P) =", res_QP["theta"])
print("cos Theta(P,Q) =", res_PQ["cos"])
print("cos Theta(Q,P) =", res_QP["cos"])


Resultado
Dis_2(P,Q) = 1.6634822817005765
Theta(P,Q) = 1.6616145512997214
Theta(Q,P) = 0.7853981633974483
cos Theta(P,Q) = -0.09069343195242567
cos Theta(Q,P) = 0.7071067811865476
